# Phase 3 — Inspect trained chesslesson HGPO model

**Training:** + tactics stages (133 task) — 学战术

**Stages included:** phase 2 + capture, protection, combat, check1, outOfCheck, checkmate1

**Checkpoint path:** `$HOME/models/chesslesson_curriculum_stage/phase3`

This notebook lets you:
1. Load the Phase 3 merged HF ckpt with vLLM
2. Sample tasks (coord + lesson) and see the exact prompt + model output
3. Step through a multi-turn lesson interactively
4. Run quick aggregate eval on a sample subset

> Run from `verl-agent-vam-agent` directory (or update REPO path in cell 2).


In [ ]:
# Imports + config
import os, sys, json, re
from pathlib import Path

REPO = Path(os.environ.get("VERL_AGENT_REPO", "/home/y50047367/chess_self_play/verl-agent-vam-agent"))
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "chess_game" / "chesslesson"))

PHASE = 3
MODEL_PATH = os.path.expanduser(f"~/models/chesslesson_curriculum_stage/phase3")
assert Path(MODEL_PATH).exists(), f"ckpt missing: {MODEL_PATH}  (Phase 3 may not be trained yet)"
print(f"Phase {PHASE} ckpt: {MODEL_PATH}")


## 1. Load chesslesson task pool

In [ ]:
# Load lesson + coord tasks (same as training env)
lessons = [json.loads(l) for l in (REPO / "chess_game/chesslesson/instructions.jsonl").open()]
coords  = [json.loads(l) for l in (REPO / "chess_game/chesslesson/coordinates.jsonl").open()]
coord_holdout = [json.loads(l) for l in (REPO / "chess_game/chesslesson/coordinates_holdout.jsonl").open()]
for r in lessons:  r["kind"] = "lesson"
for r in coords:   r["kind"] = "coordinate"
for r in coord_holdout: r["kind"] = "coordinate"

print(f"lessons:         {len(lessons)}")
print(f"coords (train):  {len(coords)}")
print(f"coords (holdout):{len(coord_holdout)}")

from reward import SPECS  # noqa
print(f"reward specs:    {len(SPECS)}  (lesson ids that have a reward spec)")


## 2. Load model with vLLM (~30s for 7B)

In [ ]:
# Load model
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
llm = LLM(
    model=MODEL_PATH,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.85,
    max_model_len=8192,
    trust_remote_code=True,
    dtype="bfloat16",
)
sp = SamplingParams(temperature=0.6, top_p=0.95, max_tokens=512, n=1)
print("loaded")


## 3. Prompt builder + action parser (same as training)

In [ ]:
# Prompt + projection helpers (mirror chesslesson env)
from chess_game.prompts_shared import (
    build_puzzle_prompt, build_lesson_initial_obs, build_lesson_step_obs,
)
from stepper import LessonStepper

_ACTION_RE = re.compile(r"<action>\s*(.*?)\s*</action>", re.DOTALL | re.IGNORECASE)

def parse_action(text):
    if "<think>" not in text or "</think>" not in text: return ""
    m = _ACTION_RE.search(text)
    return m.group(1).strip().lower().replace(" ", "") if m else ""

def render_chat(user_msg):
    return tok.apply_chat_template(
        [{"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True,
    )


## 4. Single-turn coord example (3 modes, both views)

In [ ]:
# Try 3 coord modes, 1 task each
import random
random.seed(0)

coord_samples = []
for mode in ["findSquare", "nameSquare", "squareColor"]:
    candidates = [c for c in coords if c.get("mode") == mode]
    coord_samples.append(random.choice(candidates))

prompts = [build_puzzle_prompt(c) for c in coord_samples]
rendered = [render_chat(p) for p in prompts]
outs = llm.generate(rendered, sp)

for c, p, out in zip(coord_samples, prompts, outs):
    response = out.outputs[0].text
    action = parse_action(response)
    gold = str(c["meta"]["answer"]).strip().lower()
    print("=" * 80)
    print(f"id={c['id']}  mode={c['mode']}  gold={gold!r}")
    print(f"--- USER PROMPT ---")
    print(p)
    print(f"--- MODEL RESPONSE ---")
    print(response[:600])
    print(f"--- PARSE ---")
    print(f"parsed_action={action!r}  correct={action == gold}")
    print()


## 5. Holdout coord (unseen squares — true generalization)

In [ ]:
# Run on 10 holdout coord
sample = coord_holdout[:10]
prompts = [build_puzzle_prompt(c) for c in sample]
outs = llm.generate([render_chat(p) for p in prompts], sp)

correct = 0
for c, out in zip(sample, outs):
    action = parse_action(out.outputs[0].text)
    gold = str(c["meta"]["answer"]).strip().lower()
    ok = action == gold
    correct += ok
    print(f"{c['id']:35s}  pred={action!r:12s}  gold={gold!r:10s}  {'✓' if ok else '✗'}")
print(f"\n→ {correct}/{len(sample)} holdout coord correct")


## 6. Multi-turn lesson rollout (rook-3, 3 turn) — step-by-step

In [ ]:
# Step a multi-turn lesson through env
lesson = next(l for l in lessons if l['id'] == 'rook-3')
spec = SPECS['rook-3']
stepper = LessonStepper(spec)

messages = []
obs = build_lesson_initial_obs(lesson, render_fen=stepper.board_fen(),
                               opening_opponent=list(stepper.opening_opponent) or None)
messages.append({"role": "user", "content": obs})

for turn in range(1, 9):  # max_steps=8
    if stepper.done: break
    print("=" * 80)
    print(f"TURN {turn} — USER OBS")
    print("-" * 80)
    print(messages[-1]['content'])
    
    # model generates
    rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = llm.generate([rendered], sp)
    response = out[0].outputs[0].text
    action = parse_action(response)
    messages.append({"role": "assistant", "content": response})
    
    print(f"\nTURN {turn} — MODEL RESPONSE")
    print("-" * 80)
    print(response[:400])
    print(f"\nparsed_action: {action!r}")
    
    # env.step
    pre_items = set(stepper.items)
    pre_hist  = len(stepper.chess.history)
    try:
        stepper.step(action)
    except Exception as e:
        print(f"⚠️ env rejected: {e}")
        break
    post_items = set(stepper.items)
    collected = next(iter(pre_items - post_items)) if (pre_items - post_items) else None
    new_hist = stepper.chess.history[pre_hist:]
    opp_moves = new_hist[1:] if len(new_hist) > 1 else None
    
    if not stepper.done and turn < 8:
        next_obs = build_lesson_step_obs(
            render_fen=stepper.board_fen(),
            opponent_moves=opp_moves,
            apples_left=sorted(post_items) if post_items else None,
            collected=collected,
        )
        messages.append({"role": "user", "content": next_obs})

print("=" * 80)
print(f"FINAL  vm.completed={stepper.vm.get('completed')}  items_left={sorted(stepper.items)}")
print(f"REWARD: {1.0 if stepper.vm.get('completed') else 0.0}")


## 7. Quick aggregate eval — coord training set (60)

In [ ]:
# Run all 60 coord tasks in batch
prompts = [build_puzzle_prompt(c) for c in coords]
outs = llm.generate([render_chat(p) for p in prompts], sp)

by_mode = {"findSquare": [0,0], "nameSquare": [0,0], "squareColor": [0,0]}
for c, out in zip(coords, outs):
    action = parse_action(out.outputs[0].text)
    gold = str(c["meta"]["answer"]).strip().lower()
    by_mode[c["mode"]][1] += 1
    if action == gold: by_mode[c["mode"]][0] += 1

print("Per-mode coord acc (training set):")
total_c, total_n = 0, 0
for m, (c, n) in by_mode.items():
    print(f"  {m:14s}  {c}/{n}  = {c/max(n,1):.3f}")
    total_c += c; total_n += n
print(f"  {'OVERALL':14s}  {total_c}/{total_n}  = {total_c/max(total_n,1):.3f}")


## 8. Quick aggregate eval — coord HOLDOUT (60 unseen squares)

In [ ]:
# Run all 60 holdout coord tasks
prompts = [build_puzzle_prompt(c) for c in coord_holdout]
outs = llm.generate([render_chat(p) for p in prompts], sp)

by_mode = {"findSquare": [0,0], "nameSquare": [0,0], "squareColor": [0,0]}
for c, out in zip(coord_holdout, outs):
    action = parse_action(out.outputs[0].text)
    gold = str(c["meta"]["answer"]).strip().lower()
    by_mode[c["mode"]][1] += 1
    if action == gold: by_mode[c["mode"]][0] += 1

print("Per-mode coord acc (HOLDOUT — unseen squares):")
total_c, total_n = 0, 0
for m, (c, n) in by_mode.items():
    print(f"  {m:14s}  {c}/{n}  = {c/max(n,1):.3f}")
    total_c += c; total_n += n
print(f"  {'OVERALL':14s}  {total_c}/{total_n}  = {total_c/max(total_n,1):.3f}")
